# Create KAW Wallenberg Awards (GRANT PATTERN, method-5 static HTML via Drupal)

Knut och Alice Wallenbergs Stiftelse (KAW) research projects from the foundation's own Drupal-published archive at kaw.wallenberg.org. Sweden's largest private research funder (~SEK 2-3 billion/year to basic research in medicine, technology, and natural sciences). Programs include Wallenberg Scholars, Wallenberg Academy Fellows, Wallenberg Clinical Scholars, prolongation grants, Project grants.

**Prerequisites:**
- Run `scripts/local/wallenberg_to_s3.py` first.

**Data source:** Drupal sitemap at kaw.wallenberg.org/sitemap.xml (2 paged sub-sitemaps, 2,757 total URLs) filtered to the `/en/research/{slug}` paths → **655 unique research project pages**. Each page is server-rendered with labeled paragraphs (Principal investigator, Institution, Grant in SEK, etc.) + a JSON-LD Article block.

**S3 location:** `s3a://openalex-ingest/awards/wallenberg/wallenberg_projects.parquet`

**Awarding body:**
- funder_id: 4320322327
- display_name: "Knut och Alice Wallenbergs Stiftelse"
- country: SE
- ROR: none
- DOI: 10.13039/501100004063

**Coverage (from 2026-05-22 local `--skip-upload` full run):**
- 655 project rows, year range 2011-2026
- 100% title / slug / program_label / award_year / description
- 95.3% institution
- 37.9% structured PI name (Principal investigator field present)
- 29.0% structured amount (Grant in SEK field present, total SEK 7.55B across 190 projects with amount; min SEK 10.8M, median SEK 33M, max SEK 950M)
- 100% currency = SEK for rows with an amount
- 57.7% research_field
- 33.6% co-investigators (raw string; KAW concatenates co-PI names without separators)

Older project pages (mostly pre-2014 Wallenberg Scholars batches) use the labeled-paragraph format consistently; newer batches drop some labels in favor of narrative-only descriptions. The notebook treats partial coverage as expected and waives §6.7 amount-coverage for pages that don't publish a per-project amount.

**Amount/currency:** Populated where the source page exposes a "Grant in SEK" line; NULL otherwise. **NOT a global §6.7 waiver** — KAW does publish amounts for many projects. The 29% amount coverage reflects the source's own publishing inconsistency, not a contractor-side decision.

**Provenance:** `kaw_wallenberg_projects` (direct from foundation, not an aggregator).

**Priority:** 111 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 / Wenner-Gren 110 are immediately-prior slots in flight; Vilcek at 105 is the current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wallenberg_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wallenberg/wallenberg_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.wallenberg_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.wallenberg_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

KAW DOES publish per-project amounts on some pages — the upload script extracts them where present. Runbook §1.5 RLIKE money scan should find the `amount` and `currency` columns produced by the script.

In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'wallenberg_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'wallenberg_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT title, program_label, institution, principal_investigator_raw,
       amount, currency, award_year, SUBSTR(description, 1, 150) AS desc_preview
FROM openalex.awards.wallenberg_raw LIMIT 5;


In [ ]:
%sql
-- Amount distribution sanity check.
-- Expected from local validation: 190/655 rows have amount, SEK only, min SEK 10.8M, max SEK 950M, total SEK 7.55B.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS rows_parseable_as_number,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e9, 2) AS total_sek_billions
FROM openalex.awards.wallenberg_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.wallenberg_raw;


In [ ]:
%sql
SELECT award_year, COUNT(*) as projects
FROM openalex.awards.wallenberg_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year DESC;


In [ ]:
%sql
-- Top institutions
SELECT institution, COUNT(*) as cnt
FROM openalex.awards.wallenberg_raw
GROUP BY institution ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
-- Currency check on amount-bearing rows
SELECT currency, COUNT(*) as cnt
FROM openalex.awards.wallenberg_raw
WHERE amount IS NOT NULL
GROUP BY currency;


## Step 1.6: Fail-fast — Verify the KAW Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322327;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wallenberg_awards
USING delta
AS
WITH
kaw_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322327
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.wallenberg_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        g.currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        CASE
            WHEN LOWER(COALESCE(g.program_label, '')) RLIKE 'fellow|scholar|postdoc|stipend' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,

        -- funder_scheme = the program label (e.g., "Wallenberg Academy Fellow, prolongation grant 2020")
        COALESCE(NULLIF(TRIM(g.program_label), ''), 'KAW Research Project') as funder_scheme,

        'kaw_wallenberg_projects' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator: PI name when extracted; institution always populated when present
        CASE
            WHEN g.pi_given_name IS NULL AND g.pi_family_name IS NULL AND g.institution IS NULL THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(g.pi_given_name), '') as given_name,
                NULLIF(TRIM(g.pi_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.institution), '') as name,
                    'SE' as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN kaw_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 111

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kaw_wallenberg_projects' AND priority = 111;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    111 as priority  -- KAW priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.wallenberg_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.wallenberg_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.wallenberg_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi_struct,
    COUNT(landing_page_url) as has_url,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi
FROM openalex.awards.wallenberg_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       amount, currency,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       lead_investigator.affiliation.name AS pi_affiliation,
       landing_page_url
FROM openalex.awards.wallenberg_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.wallenberg_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.wallenberg_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- §6.7 Amount/currency check. Expected: 29% pct_amount, SEK only, total ~SEK 7.5B.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt,
    ROUND(SUM(amount) / 1e9, 2) AS total_sek_billions
FROM openalex.awards.wallenberg_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.wallenberg_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.wallenberg_awards;
